# 06 Sequential Paper Dataset Runner

Run the paper datasets one after another: `sst2 -> qqp -> mnli-m`. This is the notebook to use when you want a single ordered run instead of opening one notebook per dataset.

In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from text_distillation.data import get_dataset_info, get_train_eval_splits, load_text_classification_dataset, make_tiny_subset
from text_distillation.data.datasets import get_label_names
from text_distillation.distillation import select_kcenter_embeddings, select_kcenter_tfidf, select_stratified_random
from text_distillation.model import train_text_classifier
from text_distillation.saving import create_run_dir, save_distilled_dataset, save_experiment_config, save_metrics
from text_distillation.utils import get_git_commit_hash, set_seed


In [ ]:
EXPERIMENT_PREFIX = "paper_sequential"
DATASET_NAMES = ["sst2", "qqp", "mnli-m"]

# Paper-style coreset baselines. In the paper, Random means DPC random samples per class.
METHOD_NAMES = ["random", "kcenter_cls"]
DPC_VALUES = [5, 10, 20]

# Full-data runs are expensive on QQP/MNLI. Enable when you want to measure full-dataset references yourself.
RUN_FULL_DATA_BASELINES = False

MODEL_NAME = "bert-base-uncased"
EMBEDDING_MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
EMBEDDING_BATCH_SIZE = 128
TFIDF_MAX_FEATURES = 50_000

NUM_TRAIN_EPOCHS = 3.0
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# T4 16GB starting point. If CUDA OOM happens, lower TRAIN_BATCH_SIZE to 32.
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
GRADIENT_ACCUMULATION_STEPS = 1
MIXED_PRECISION = "auto"
DATALOADER_NUM_WORKERS = 2
SEED = 42

# Keep None for real runs. Use tiny values to debug the whole notebook quickly.
MAX_TRAIN_POOL_DPC = None
MAX_EVAL_SAMPLES = None

set_seed(SEED)


In [ ]:
def maybe_make_tiny_train_pool(train_dataset, dataset_info):
    if MAX_TRAIN_POOL_DPC is None:
        return train_dataset
    return make_tiny_subset(
        train_dataset,
        n_per_class=MAX_TRAIN_POOL_DPC,
        label_column=dataset_info.label_column,
        seed=SEED,
    )


def maybe_make_tiny_eval(eval_dataset):
    if MAX_EVAL_SAMPLES is None:
        return eval_dataset
    return make_tiny_subset(eval_dataset, total_size=MAX_EVAL_SAMPLES, seed=SEED)


def build_distilled_dataset(method_name, train_dataset, dataset_info, dpc, seed):
    if method_name == "random":
        return select_stratified_random(
            train_dataset,
            label_column=dataset_info.label_column,
            k_per_class=dpc,
            seed=seed,
        )
    if method_name == "kcenter_cls":
        return select_kcenter_embeddings(
            train_dataset,
            text_columns=dataset_info.text_columns,
            label_column=dataset_info.label_column,
            k_per_class=dpc,
            model_name=EMBEDDING_MODEL_NAME,
            batch_size=EMBEDDING_BATCH_SIZE,
            max_length=MAX_LENGTH,
            seed=seed,
        )
    if method_name == "kcenter_tfidf":
        return select_kcenter_tfidf(
            train_dataset,
            text_columns=dataset_info.text_columns,
            label_column=dataset_info.label_column,
            k_per_class=dpc,
            seed=seed,
            max_features=TFIDF_MAX_FEATURES,
        )
    raise ValueError(f"Unknown method: {method_name}")


In [ ]:
def train_and_save(
    *,
    experiment_name,
    dataset_name,
    dataset_info,
    train_dataset,
    eval_dataset,
    label_names,
    num_labels,
    method_name,
    dpc=None,
    full_train_size=None,
    train_pool_size=None,
    selection_time_sec=0.0,
):
    run_dir = create_run_dir(PROJECT_ROOT / "artifacts" / "runs", experiment_name)
    config = {
        "experiment_name": experiment_name,
        "dataset_name": dataset_name,
        "method_name": method_name,
        "model_name": MODEL_NAME,
        "embedding_model_name": EMBEDDING_MODEL_NAME if method_name == "kcenter_cls" else None,
        "dpc": dpc,
        "k_total": len(train_dataset),
        "text_columns": dataset_info.text_columns,
        "label_column": dataset_info.label_column,
        "metric_name": dataset_info.metric_name,
        "train_split": dataset_info.train_split,
        "eval_split": dataset_info.eval_split,
        "max_length": MAX_LENGTH,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "mixed_precision": MIXED_PRECISION,
        "dataloader_num_workers": DATALOADER_NUM_WORKERS,
        "seed": SEED,
        "full_train_size": full_train_size,
        "train_pool_size": train_pool_size,
        "train_size": len(train_dataset),
        "eval_size": len(eval_dataset),
        "compression_ratio": full_train_size / len(train_dataset) if full_train_size is not None else None,
        "selection_time_sec": selection_time_sec,
        "git_commit": get_git_commit_hash(PROJECT_ROOT),
    }
    save_experiment_config(config, run_dir)
    if method_name != "full_data":
        save_distilled_dataset(train_dataset, run_dir)

    train_start = time.perf_counter()
    _, metrics = train_text_classifier(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        model_name=MODEL_NAME,
        output_dir=run_dir,
        text_columns=dataset_info.text_columns,
        label_column=dataset_info.label_column,
        num_labels=num_labels,
        label_names=label_names,
        max_length=MAX_LENGTH,
        metric_name=dataset_info.metric_name,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        train_batch_size=TRAIN_BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        mixed_precision=MIXED_PRECISION,
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        seed=SEED,
    )
    metrics["training_time_sec"] = time.perf_counter() - train_start
    metrics["selection_time_sec"] = selection_time_sec
    save_metrics(metrics, run_dir)
    return {**config, **metrics}


In [ ]:
results = []

for dataset_name in DATASET_NAMES:
    print(f"\n=== Dataset: {dataset_name} ===")
    dataset_info = get_dataset_info(dataset_name)
    dataset = load_text_classification_dataset(dataset_name)
    full_train_dataset, eval_dataset = get_train_eval_splits(dataset, dataset_name)
    train_pool = maybe_make_tiny_train_pool(full_train_dataset, dataset_info)
    eval_dataset = maybe_make_tiny_eval(eval_dataset)

    label_names = get_label_names(full_train_dataset, dataset_info.label_column)
    num_labels = len(label_names) if label_names is not None else len(set(full_train_dataset[dataset_info.label_column]))

    if RUN_FULL_DATA_BASELINES:
        experiment_name = f"{EXPERIMENT_PREFIX}_{dataset_name}_full_data_bert_base"
        row = train_and_save(
            experiment_name=experiment_name,
            dataset_name=dataset_name,
            dataset_info=dataset_info,
            train_dataset=train_pool,
            eval_dataset=eval_dataset,
            label_names=label_names,
            num_labels=num_labels,
            method_name="full_data",
            full_train_size=len(full_train_dataset),
            train_pool_size=len(train_pool),
        )
        results.append(row)
        print(dataset_name, "full_data", {k: row[k] for k in row if k in ["score", "accuracy", "f1", "f1_macro"]})

    for dpc in DPC_VALUES:
        for method_name in METHOD_NAMES:
            print(f"Running {dataset_name} | {method_name} | DPC={dpc}")
            experiment_name = f"{EXPERIMENT_PREFIX}_{dataset_name}_{method_name}_bert_base_dpc{dpc}"
            selection_start = time.perf_counter()
            distilled_dataset = build_distilled_dataset(method_name, train_pool, dataset_info, dpc, SEED)
            selection_time_sec = time.perf_counter() - selection_start

            row = train_and_save(
                experiment_name=experiment_name,
                dataset_name=dataset_name,
                dataset_info=dataset_info,
                train_dataset=distilled_dataset,
                eval_dataset=eval_dataset,
                label_names=label_names,
                num_labels=num_labels,
                method_name=method_name,
                dpc=dpc,
                full_train_size=len(full_train_dataset),
                train_pool_size=len(train_pool),
                selection_time_sec=selection_time_sec,
            )
            results.append(row)
            print(dataset_name, method_name, "DPC", dpc, {k: row[k] for k in row if k in ["score", "accuracy", "f1", "f1_macro"]})

results
